# jepa-as-a-tool: train the world model (v2)

Same steps as v1, with more data and longer imagined chains:

| | v1 | v2 |
|---|---|---|
| Episodes | 4,000 | **12,000** |
| Pushes per episode | 6 | **8** |
| Pushes chained per training sample | 3 | **5** |
| Predictor training steps | 20,000 | **50,000** |

Needs ~16 GB of free disk. ~25 GB of RAM lets training keep the latents in memory;
with less (Colab free: ~12 GB) it reads them from disk, which works but is slower.
At the end you download `runs/wm-v2` and send it back. Total: ~4–5 h.

**Where to run it:** any notebook with a GPU (≥ 15 GB) and internet.
- **Kaggle**: Accelerator *GPU T4 x1* (or P100), Internet *on*, Persistence *Files only*.
- **Colab**: Runtime → *T4 GPU*. Files in `/content` vanish when the runtime stops.
- **Elsewhere**: files go to the folder the notebook starts in.

Only one GPU is used: a second one would sit idle.

**Long run tip (Kaggle):** *Save Version → Save & Run All* runs every cell in the background,
even with the browser closed (12 h max). The output files appear in the version's Output tab.

In [ ]:
import os

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    WORK = os.getcwd()
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .

If pip printed that it upgraded **numpy** or **torch**, restart the kernel, then run the cell above again. pip's complaints about other packages (gradio, bigframes, ...) are harmless.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h {WORK} | tail -1
!git log --oneline -1

Free disk: v1's latents are no longer needed (v1's model in `runs/wm` stays, for the comparison in step 5).

In [ ]:
!rm -f data/random-4k/latents.npy

## 1. Data (CPU, ~5–10 min)

12,000 episodes of 8 random pushes = 108,000 poses. Prints progress every 10%.

In [ ]:
!jat gen data/random-12k --episodes 12000 --steps 8 --workers 4

## 2. V-JEPA 2 latents (GPU, ~40–80 min)

Writes `data/random-12k/latents.npy`, ~14 GB.

In [ ]:
!jat encode data/random-12k

## 3. Translator (GPU, ~10 min)

Should land around v1's: median ~3 px and ~1.5° on test frames.

In [ ]:
!jat train translator data/random-12k --out runs/wm-v2 --steps 8000

## 4. Predictor (GPU, ~2.5–3.5 h)

Prints the time left as it trains. Watch `val 1 push`: v1 ended around 4–5 px.
The final table now goes up to 8 pushes.

In [ ]:
!jat train predictor data/random-12k --out runs/wm-v2 --steps 50000 --rollout 5

## 5. v1 on the same test episodes

The table above is v2. This one is v1 on the same v2 test episodes: a fair comparison.

In [ ]:
if os.path.isdir("runs/wm"):
    !jat wm-eval data/random-12k --wm runs/wm
else:
    print("runs/wm (v1) is not here; skip.")

## 6. Send back

Download `wm-v2.zip` and send it, with the output of steps 3, 4 and 5.

In [ ]:
!cd runs && zip -r {WORK}/wm-v2.zip wm-v2
!ls -lh {WORK}/wm-v2.zip